# Napari IMC Image and Segmentation Viewer

This notebook opens preprocessed IMC images and matching Mesmer segmentation masks in Napari. The notebook is configured for the current project structure and automatically reads available ROI images from `data/img/`, masks from `data/masks/`, and marker names from `data/panel.csv`.

Visualization purpose:

- inspect marker-channel signal quality in the original multi-channel TIFF image
- inspect whether Mesmer cell masks align with visible cellular structures
- compare nuclear, membrane, lineage, and plasma-cell-associated marker channels
- support manual quality review before downstream interpretation

Napari is an interactive desktop viewer. The notebook must be run in a local Jupyter environment with GUI support.

## Step 0: Configure Interactive Napari Backend

Napari uses a Qt graphical interface. The `%gui qt` command enables the Qt event loop inside Jupyter so the Napari window remains responsive while notebook cells continue to run.

In [2]:
%gui qt


## Step 1: Import Libraries and Locate the Workflow Directory

The workflow directory is detected automatically. If the notebook is opened from `notebooks/`, the parent directory is used. If the notebook is opened from the project root, the current directory is used.

Required Python packages:

```text
napari
pandas
tifffile
numpy
```


In [3]:
from pathlib import Path

import napari
import numpy as np
import pandas as pd
import tifffile

current = Path.cwd().resolve()
if current.name == 'notebooks':
    WORKFLOW_DIR = current.parent
else:
    WORKFLOW_DIR = current

DATA_DIR = WORKFLOW_DIR / 'data'
IMG_DIR = DATA_DIR / 'img'
MASK_DIR = DATA_DIR / 'masks'
PANEL_FILE = DATA_DIR / 'panel.csv'

print('Workflow directory:', WORKFLOW_DIR)
print('Image directory exists:', IMG_DIR.exists())
print('Mask directory exists:', MASK_DIR.exists())
print('Panel file exists:', PANEL_FILE.exists())


Workflow directory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction
Image directory exists: True
Mask directory exists: True
Panel file exists: True


## Step 2: Read Marker Names from the Panel

The panel table links each image channel to a biological marker name. These names are used as Napari layer names so individual channels can be switched on and off during visual inspection.

Only channels with `keep == 1` are used, matching the Steinbock-compatible analysis panel.

In [4]:
if not PANEL_FILE.exists():
    raise FileNotFoundError(f'Panel file not found: {PANEL_FILE}')

panel = pd.read_csv(PANEL_FILE)
panel = panel.loc[panel['keep'].astype(str) == '1'].copy()
channel_names = panel['name'].astype(str).tolist()

print(f'Markers available for visualization: {len(channel_names)}')
display(panel[['channel', 'name', 'deepcell']].head(12))


Markers available for visualization: 42


,channel,name,deepcell
0,ArAr80,ArAr80,NaN
1,I127,I1277,NaN
2,Xe131,Xe131,NaN
3,Xe134,Xe134,NaN
4,Ba138,Ba138,NaN
5,Pr141,CD38,NaN
6,Nd142,Perilipin,NaN
7,Nd143,Vimentin,NaN
8,Nd144,B4GALT1,NaN
9,Nd145,MPO,NaN


## Step 3: Select an ROI Image

Available ROI identifiers are detected from `data/img/*.tiff`. The selected ROI must have a matching mask file in `data/masks/` for segmentation overlay.

Change `selected_image_id` to inspect a different ROI.

In [5]:
image_files = sorted(IMG_DIR.glob('*.tiff'))
if not image_files:
    raise FileNotFoundError(f'No TIFF images found in: {IMG_DIR}')

available_image_ids = [path.stem for path in image_files]
print('Available ROI images:')
for image_id in available_image_ids:
    mask_status = 'mask found' if (MASK_DIR / f'{image_id}.tiff').exists() else 'mask missing'
    print(f'  {image_id} ({mask_status})')

selected_image_id = available_image_ids[0]
print('\nSelected ROI:', selected_image_id)


Available ROI images:
  TS-373_IMC01_UB_001 (mask found)
  TS-373_IMC01_UB_002 (mask found)
  TS-373_IMC02_MGUS_001 (mask found)
  TS-373_IMC02_MGUS_002 (mask found)
  TS-373_IMC05_MGUS_001 (mask found)
  TS-373_IMC05_MGUS_002 (mask found)
  TS-373_IMC09_B_001 (mask found)
  TS-373_IMC09_B_002 (mask found)

Selected ROI: TS-373_IMC01_UB_001


## Step 4: Load Image and Mask Arrays

The IMC image is stored as a multi-channel TIFF. In this workflow, channels are expected along the first axis. The mask is stored as a two-dimensional label image where each integer label corresponds to one segmented cell object.

Data meaning:

- image pixel values represent marker intensities measured by imaging mass cytometry
- each channel corresponds to one metal-tagged antibody or acquisition channel
- mask labels represent segmented single-cell objects generated by Mesmer
- label value `0` represents background outside segmented cells


In [6]:
img_file = IMG_DIR / f'{selected_image_id}.tiff'
mask_file = MASK_DIR / f'{selected_image_id}.tiff'

if not img_file.exists():
    raise FileNotFoundError(f'Image file not found: {img_file}')
if not mask_file.exists():
    raise FileNotFoundError(f'Mask file not found: {mask_file}')

img = tifffile.imread(img_file)
mask = tifffile.imread(mask_file)

if img.ndim != 3:
    raise ValueError(f'Expected a 3D multi-channel image. Observed shape: {img.shape}')
if mask.ndim != 2:
    raise ValueError(f'Expected a 2D segmentation mask. Observed shape: {mask.shape}')

if img.shape[0] != len(channel_names) and img.shape[-1] == len(channel_names):
    img = np.moveaxis(img, -1, 0)

if img.shape[0] != len(channel_names):
    print('Warning: image channel count does not match panel marker count.')
    print('Image channels:', img.shape[0])
    print('Panel markers:', len(channel_names))

print('Image file:', img_file)
print('Image shape:', img.shape)
print('Image dtype:', img.dtype)
print('Mask file:', mask_file)
print('Mask shape:', mask.shape)
print('Segmented cell labels:', int(mask.max()))


Image file: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/img/TS-373_IMC01_UB_001.tiff
Image shape: (42, 1000, 1000)
Image dtype: float32
Mask file: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/masks/TS-373_IMC01_UB_001.tiff
Mask shape: (1000, 1000)
Segmented cell labels: 5652


## Step 5: Define Helpful Marker Groups

Marker groups make visual inspection faster. These groups do not change the data; they only control which Napari layers are visible by default.

Default visible groups:

- nuclear markers for cell nuclei
- membrane markers used for segmentation
- plasma-cell-associated markers relevant to the multiple myeloma context
- immune lineage markers for immune-cell inspection


In [7]:
marker_groups = {
    'nuclear': ['HistoneH3', '191Ir', '193Ir'],
    'membrane': ['CD45', 'CD3', 'CD98', 'CD138'],
    'plasma_cell': ['CD138', 'CD38', 'CD56', 'BCMA', 'CD319', 'CD98'],
    'immune_lineage': ['CD45', 'CD3', 'CD4', 'CD8a', 'CD11c', 'CD14', 'CD15', 'CD16', 'CD68'],
}

channel_index = {name: index for index, name in enumerate(channel_names)}
available_marker_groups = {
    group: [marker for marker in markers if marker in channel_index]
    for group, markers in marker_groups.items()
}

for group, markers in available_marker_groups.items():
    print(f'{group}: {markers}')


nuclear: ['HistoneH3', '191Ir', '193Ir']
membrane: ['CD45', 'CD3', 'CD98', 'CD138']
plasma_cell: ['CD138', 'CD38', 'CD98']
immune_lineage: ['CD45', 'CD3', 'CD4', 'CD11c', 'CD68']


## Step 6: Open Napari Viewer

Each marker channel is added as a separate image layer. The segmentation mask is added as a labels layer. Marker layers are initially hidden except selected review markers, keeping the viewer readable when many channels are present.

Useful inspection pattern:

```text
1. Turn on nuclear markers and inspect nuclear signal.
2. Turn on membrane markers and inspect membrane continuity.
3. Turn on the Cells labels layer and compare mask boundaries with marker signal.
4. Toggle lineage markers to inspect whether biological signal appears spatially plausible.
```


In [8]:
viewer = napari.Viewer(title=f'IMC visualization: {selected_image_id}')
viewer.axes.visible = True
viewer.dims.axis_labels = ('y', 'x')
viewer.scale_bar.visible = True
viewer.scale_bar.unit = 'pixel'

default_visible_markers = set(
    available_marker_groups['nuclear']
    + available_marker_groups['membrane']
    + available_marker_groups['plasma_cell']
)

for index, marker in enumerate(channel_names[: img.shape[0]]):
    channel = img[index]
    layer = viewer.add_image(
        channel,
        name=marker,
        colormap='gray',
        blending='additive',
        visible=marker in default_visible_markers,
        contrast_limits=np.percentile(channel, [1, 99.5]).tolist(),
    )

mask_layer = viewer.add_labels(
    mask,
    name='Mesmer cell masks',
    blending='translucent',
    opacity=0.35,
    visible=True,
)

print('Napari viewer opened for:', selected_image_id)
print('Visible marker layers:', sorted(default_visible_markers))


Napari viewer opened for: TS-373_IMC01_UB_001
Visible marker layers: ['191Ir', '193Ir', 'CD138', 'CD3', 'CD38', 'CD45', 'CD98', 'HistoneH3']


## Step 7: Optional Composite Review Layers

Composite layers combine selected channels into a small number of review images. These layers are useful for quick segmentation checks, while individual marker layers remain available for detailed inspection.

In [9]:
def composite(markers):
    indices = [channel_index[marker] for marker in markers if marker in channel_index and channel_index[marker] < img.shape[0]]
    if not indices:
        return None
    return np.max(img[indices], axis=0)

review_composites = {
    'Composite nuclear': composite(available_marker_groups['nuclear']),
    'Composite membrane': composite(available_marker_groups['membrane']),
    'Composite plasma-cell-associated': composite(available_marker_groups['plasma_cell']),
    'Composite immune-lineage': composite(available_marker_groups['immune_lineage']),
}

for name, array in review_composites.items():
    if array is None:
        continue
    viewer.add_image(
        array,
        name=name,
        colormap='gray',
        blending='additive',
        visible=False,
        contrast_limits=np.percentile(array, [1, 99.5]).tolist(),
    )

print('Composite review layers added.')


Composite review layers added.


## Step 8: Optional Screenshot Export

The current Napari canvas can be exported as a PNG screenshot. This is useful for documenting segmentation quality or marker localization examples.

In [10]:
screenshot_dir = WORKFLOW_DIR / 'figures' / 'napari_screenshots'
screenshot_dir.mkdir(parents=True, exist_ok=True)
screenshot_path = screenshot_dir / f'{selected_image_id}_napari_view.png'

# Run this line after arranging the Napari view as needed.
# viewer.screenshot(path=screenshot_path, canvas_only=True)

print('Screenshot output path:', screenshot_path)


Screenshot output path: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/figures/napari_screenshots/TS-373_IMC01_UB_001_napari_view.png
